# 🧬 Mega Biyolojik Turnuva (Onlarca Evrim Kombinasyonu)

Bu notebook, Biyolojik Sineğin (FlyOpt) sahip olabileceği **tüm duyu organı ve motor beceri kombinasyonlarını (18 farklı mimariyi)** tek bir döngüde test eder ve piyasa standardı PSO ile çarpıştırır.

**Denenen Özellikler (Kombinasyonlar):**
- **Koku Alma:** Kör(GPS) | Anlık Koku (Gradyan) | Koku Hafızası (Momentum)
- **Sürü Zekası:** Yalnız (Sürü Yok) | Sürü-Ana-Loba (Node1-2) | Sürü-Yan-Loba (Node3-4)
- **Enerji/Kas:** Sabit Hız | Yorulma/Odaklanma (Azalan Hız)

Görev: **Rastrigin** (Ölümcül Tuzaklar).

In [ ]:
# 1. GEREKSİNİMLER VE AĞ YÜKLEMESİ
from google.colab import drive
import sys, os, glob, time
import torch
import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

drive.mount('/content/drive')
project_path = '/content/drive/MyDrive/fly_op'
if not os.path.exists(project_path): project_path = '/content/drive/MyDrive/fly_op/fly_op'
sys.path.append(project_path); sys.path.append(os.path.join(project_path, 'src')); os.chdir(project_path)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from flyopt.variants.rate_brain import RateBrain, RateBrainConfig, build_subgraph_bfs, select_connected_encode_decode

def rastrigin(x): return 10 * x.shape[1] + torch.sum(x**2 - 10 * torch.cos(2 * np.pi * x), dim=1)

found_files = glob.glob('/content/drive/MyDrive/**/malecns_adjacency.npz', recursive=True)
DATA_PROCESSED = os.path.dirname(found_files[0]) if found_files else f"{project_path}/data/processed"
base_weights = sparse.load_npz(f"{DATA_PROCESSED}/malecns_adjacency.npz")
afferent, efferent = np.load(f"{DATA_PROCESSED}/malecns_afferent_indices.npy"), np.load(f"{DATA_PROCESSED}/malecns_efferent_indices.npy")

encode_full, decode_full = select_connected_encode_decode(base_weights, afferent, efferent, n_encode=4, n_decode=4, max_hops=6, n_encode_candidates=200, seed=9000)
sub_real, encode_idx, decode_idx, _ = build_subgraph_bfs(base_weights, encode_full, decode_full, 3000, seed=9000)
cfg = RateBrainConfig(dim=2, n_readout=len(decode_idx), T=20, decode_scale=0.5, train_gain=True)
brain = RateBrain(sub_real, encode_idx, decode_idx, cfg, seed=42).to(device)
print("✅ Beyin Yüklendi! Turnuvaya Hazır.")

In [ ]:
# 2. MEGA TURNUVA MOTORU (18 MİMARİ + PSO)
def pso_optimize(num_particles=500, iters=100):
    x = (torch.rand((num_particles, 2), device=device) * 10 - 5)
    v = torch.zeros_like(x)
    pbest, pbest_obj = x.clone(), rastrigin(x)
    gbest_obj = torch.min(pbest_obj)
    gbest = pbest[torch.argmin(pbest_obj)].clone()
    history = []
    for _ in range(iters):
        r1, r2 = torch.rand((num_particles, 2), device=device), torch.rand((num_particles, 2), device=device)
        v = 0.5 * v + 1.5 * r1 * (pbest - x) + 1.5 * r2 * (gbest - x)
        x = x + v
        obj = rastrigin(x)
        mask = obj < pbest_obj
        pbest[mask], pbest_obj[mask] = x[mask], obj[mask]
        if torch.min(pbest_obj) < gbest_obj:
            gbest_obj = torch.min(pbest_obj)
            gbest = pbest[torch.argmin(pbest_obj)].clone()
        history.append(gbest_obj.item())
    return history

def multi_variant_fly(koku_tipi, suru_tipi, enerji_tipi, num_particles=500, iters=100):
    x = (torch.rand((num_particles, 2), device=device) * 10 - 5)
    x.requires_grad_(True)
    best_obj = rastrigin(x).detach()
    gbest_obj = torch.min(best_obj)
    gbest_x = x[torch.argmin(best_obj)].detach().clone()
    momentum = torch.zeros_like(x, device=device)
    history = []
    
    for i in range(iters):
        obj = rastrigin(x)
        obj.sum().backward()
        with torch.no_grad():
            grad = x.grad.clone()
            x.grad.zero_()
            
            # 1. KOKU MODÜLÜ
            if koku_tipi == 'Kör': vec12 = x.detach() * 0.1 # GPS saçmalığı
            elif koku_tipi == 'Anlık': vec12 = torch.nn.functional.normalize(-grad, p=2, dim=1)
            elif koku_tipi == 'Hafızalı':
                momentum = 0.8 * momentum + 0.2 * grad
                vec12 = torch.nn.functional.normalize(-momentum, p=2, dim=1)
                
            # 2. SÜRÜ MODÜLÜ VE NÖRON ATAMALARI
            swarm_vec = torch.nn.functional.normalize(gbest_x - x.detach(), p=2, dim=1)
            swarm_vec = torch.nan_to_num(swarm_vec, 0.0)
            
            if suru_tipi == 'Yok': vec34 = torch.zeros_like(vec12)
            elif suru_tipi == 'Yan_Lob': vec34 = swarm_vec
            elif suru_tipi == 'Ana_Lob':
                vec12 = swarm_vec # Sürüyü ana loba (koku yerine) ver
                vec34 = torch.zeros_like(vec12)
                
            env_input = torch.cat([vec12, vec34], dim=1)
            fly_step = brain(env_input)
            
            # 3. ENERJİ MODÜLÜ
            lr = 0.1 if enerji_tipi == 'Sabit' else max(0.005, 0.2 * (1.0 - (i / iters)))
            
            x_new = x.detach() + fly_step[:, :2] * lr
            new_obj = rastrigin(x_new)
            mask = new_obj < best_obj
            
            x_updated = x.detach().clone()
            x_updated[mask] = x_new[mask]
            x = x_updated.clone()
            x.requires_grad_(True)
            best_obj[mask] = new_obj[mask]
            
            if torch.min(best_obj) < gbest_obj:
                gbest_obj = torch.min(best_obj)
                gbest_x = x[torch.argmin(best_obj)].detach().clone()
            history.append(gbest_obj.item())
    return history

In [ ]:
# 3. TURNUVAYI BAŞLAT
koku_tipleri = ['Kör', 'Anlık', 'Hafızalı']
suru_tipleri = ['Yok', 'Yan_Lob', 'Ana_Lob']
enerji_tipleri = ['Sabit', 'Yorulan']

results = {}
print("🧬 MEGA TURNUVA BAŞLADI (18 Sinek Mimarisi + PSO)")
print("Her mimari 500 ajanla Rastrigin'de ölüm kalım savaşı veriyor...\n")

results['0_PSO_Klasik'] = pso_optimize()
print(f"PSO_Klasik tamamlandı. Final Skoru: {results['0_PSO_Klasik'][-1]:.6f}")

for k in koku_tipleri:
    for s in suru_tipleri:
        for e in enerji_tipleri:
            # Mantıksız kombinasyonları ele (örn: Kör olup koku hafızası olmaz)
            name = f"Fly_{k}_{s}_{e}"
            hist = multi_variant_fly(k, s, e)
            results[name] = hist
            print(f"{name} tamamlandı. Final Skoru: {hist[-1]:.6f}")

# Sonuçları DataFrame yap ve kaydet
df_mega = pd.DataFrame(results)
df_mega.insert(0, 'Iterasyon', range(1, 101))
csv_name = 'Mega_Turnuva_Sonuclari.csv'
df_mega.to_csv(csv_name, index=False)

In [ ]:
# 4. EN İYİ 5 MİMARİYİ BUL VE ÇİZDİR
final_scores = df_mega.iloc[-1].drop('Iterasyon')
top5_names = final_scores.nsmallest(6).index.tolist() # Kendisiyle PSO + En iyi 5 sinek
if '0_PSO_Klasik' not in top5_names: top5_names.append('0_PSO_Klasik')

plt.figure(figsize=(14, 8))
sns.set_theme(style="darkgrid")

colors = sns.color_palette("husl", len(top5_names))
for i, col in enumerate(top5_names):
    linewidth = 3 if col == '0_PSO_Klasik' else 2
    linestyle = '--' if col == '0_PSO_Klasik' else '-'
    plt.plot(df_mega['Iterasyon'], df_mega[col], label=f"{col} ({df_mega[col].iloc[-1]:.4f})", 
             color=colors[i], linewidth=linewidth, linestyle=linestyle)

plt.yscale('log')
plt.title('Mega Biyolojik Turnuva: En İyi 5 Evrimleşmiş Sinek vs PSO (Rastrigin)')
plt.xlabel('İterasyon')
plt.ylabel('Hata Payı (Log Scale)')
plt.legend()
grafik_name = 'Mega_Turnuva_Top5.png'
plt.savefig(grafik_name, dpi=300)
plt.show()

print("\n🏆 İŞLEM BİTTİ! Tüm denemelerin sonuçları ve grafik bilgisayara iniyor...")
try:
    files.download(csv_name)
    files.download(grafik_name)
except:
    print("Manuel indirebilirsiniz.")